# 🧪 [Day 32] 대용량 지식그래프 구축·멱등 적재(ETL 파이프라인) 실전 워크북

- **과정 구분**: 지식그래프 엔지니어링 실전 마스터
- **데이터셋**: [DART-Trace] 3,746건 대량 공시 지분 청크 & [ART:READY] 15개교 미대 요강 청크
- **핵심 미션**: UNWIND + MERGE 기반 청크 트랜잭션 적재, 2회차 재실행 시 노드 순증 0건(Idempotency) 검증을 직접 실습한다.

## 1. 환경 설정 및 드라이버 연결

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USERNAME", os.getenv("NEO4J_USER", "neo4j"))
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print("✅ Neo4j 연결 성공:", NEO4J_URI)

## 2. [DART-Trace] 공시 지분 및 증거 노드 청크 멱등 적재

In [ ]:
dart_chunk = [
    {"corp_code": "00126380", "corp_name": "삼성전자", "market_type": "KOSPI", "holder_key": "HOLDER_NPS", "holder_name": "국민연금공단", "holder_type": "기관투자자", "stake_ratio": 7.25, "base_date": "2024-03-31", "fragment_id": "FRAG_202403310001", "rcept_no": "202403310001", "table_xpath": "table[3]/tr[5]"},
    {"corp_code": "00126380", "corp_name": "삼성전자", "market_type": "KOSPI", "holder_key": "HOLDER_SAMSUNG_LIFE", "holder_name": "삼성생명보험", "holder_type": "계열회사", "stake_ratio": 8.51, "base_date": "2024-03-31", "fragment_id": "FRAG_202403310002", "rcept_no": "202403310001", "table_xpath": "table[3]/tr[6]"}
]

dart_load_cypher = """
UNWIND $batch AS row
MERGE (c:Company {corp_code: row.corp_code})
  ON CREATE SET c.name = row.corp_name, c.market_type = row.market_type
MERGE (s:Shareholder {holder_key: row.holder_key})
  ON CREATE SET s.name = row.holder_name, s.holder_type = row.holder_type
MERGE (s)-[r:HOLDS_ECONOMIC_STAKE]->(c)
  ON CREATE SET r.stake_ratio = row.stake_ratio, r.base_date = row.base_date
MERGE (e:EvidenceFragment {fragment_id: row.fragment_id})
  ON CREATE SET e.rcept_no = row.rcept_no, e.table_xpath = row.table_xpath
MERGE (c)-[:BACKED_BY_EVIDENCE]->(e);
"""

with driver.session() as session:
    session.run(dart_load_cypher, batch=dart_chunk)
    print("✅ DART 공시 청크 멱등 적재 완료")

## 3. [ART:READY] 전국 미대 전형 요강 청크 멱등 적재

In [ ]:
art_chunk = [
    {"univ_code": "CAU_SEOUL", "univ_name": "중앙대학교", "campus": "서울", "track_id": "CAU_2027_PRACTICAL", "track_name": "2027 수시 실기형", "season": "수시", "practical_code": "PRACT_SKETCH", "practical_name": "소묘", "category": "회화", "stage": 1, "ratio": 80.0}
]

art_load_cypher = """
UNWIND $batch AS row
MERGE (u:University {univ_code: row.univ_code})
  ON CREATE SET u.name = row.univ_name, u.campus = row.campus
MERGE (t:AdmissionTrack {track_id: row.track_id})
  ON CREATE SET t.name = row.track_name, t.season = row.season
MERGE (p:PracticalType {code: row.practical_code})
  ON CREATE SET p.name = row.practical_name, p.category = row.category
MERGE (u)-[:OFFERS_TRACK]->(t)
MERGE (t)-[r:REQUIRES_PRACTICAL]->(p)
  ON CREATE SET r.stage = row.stage, r.ratio = row.ratio;
"""

with driver.session() as session:
    session.run(art_load_cypher, batch=art_chunk)
    print("✅ ART:READY 전형 청크 멱등 적재 완료")

## 4. [검증] 재실행 시 노드/관계 순증 0건(100% 멱등성) 실측 검증

In [ ]:
count_cypher = "MATCH (n) WITH count(n) AS node_count MATCH ()-[r]->() RETURN node_count, count(r) AS rel_count;"

with driver.session() as session:
    cnt1 = session.run(count_cypher).single()
    print(f"1회차 노드: {cnt1['node_count']}개, 관계: {cnt1['rel_count']}개")
    
    # 2회차 재실행
    session.run(dart_load_cypher, batch=dart_chunk)
    session.run(art_load_cypher, batch=art_chunk)
    
    cnt2 = session.run(count_cypher).single()
    print(f"2회차 노드: {cnt2['node_count']}개, 관계: {cnt2['rel_count']}개")
    
    delta_nodes = cnt2['node_count'] - cnt1['node_count']
    delta_rels = cnt2['rel_count'] - cnt1['rel_count']
    print(f"순증 노드: {delta_nodes}개, 순증 관계: {delta_rels}개 ➔ {'✅ 멱등성 합격' if delta_nodes == 0 and delta_rels == 0 else '❌ 실패'}")